# U07_C1 — Distances and Trees
**Course:** Concepts of Bioinformatics (CBBI-501) · **Unit 7:** Phylogenetic Analysis
**Type:** Course notebook · **Time:** 40 min · **Difficulty:** ★★☆
**Goes with:** Lectures L18–L20, Practical S8 (Tree extension)
**Before this:** U06_C1

## 1. What you will learn
- **Explain** core phylogenetic terminology: nodes, branches, clades, roots, and outgroups [LO5].
- **Calculate** pairwise evolutionary distance matrices from multiple sequence alignments [LO5].
- **Perform** UPGMA clustering step-by-step by hand and with Biopython's `Bio.Phylo` [LO5].
- **Differentiate** orthologues (speciation events) from paralogues (gene duplication events) on a gene tree [LO5].


## 2. Biological question
Are myoglobin (*MB*) and hemoglobin beta (*HBB*) orthologues or paralogues? When we construct a phylogenetic tree from human globin sequences, we observe that hemoglobin beta (*HBB*) and delta (*HBD*) cluster together with nearly identical sequences, while myoglobin branches much earlier. How do we reconstruct the evolutionary branching history of these genes from sequence distances?


In [ ]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)

NOTEBOOK_ID = "U07_C1"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs" / NOTEBOOK_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

import Bio
print("Python", sys.version.split()[0], "| Biopython", Bio.__version__)

## 3. Concept 1: Phylogenetic Tree Anatomy and Orthology
### Theory
A phylogenetic tree is a mathematical graph modeling evolutionary relationships:
- **Terminal Nodes (Leaves / Taxa)**: Extant observed sequences.
- **Internal Nodes**: Hypothetical common ancestors.
- **Branches**: Evolutionary lineages; branch length represents inferred evolutionary change (divergence time or substitutions per site).
- **Clade (Monophyletic Group)**: An ancestor and *all* of its descendants.
- **Rooted vs. Unrooted**: A rooted tree specifies the direction of evolutionary time from an ultimate common ancestor (often defined using an **outgroup**).

**Orthologues vs. Paralogues:**
- **Orthologues**: Genes separated by a **speciation** event (e.g. human HBB vs chimp HBB). They typically perform identical biological functions in different species.
- **Paralogues**: Genes separated by an internal **gene duplication** event (e.g. human HBB and human HBD, or HBB and Myoglobin). Paralogues often evolve new or specialized functions (neofunctionalization / subfunctionalization).


## 4. Concept 2: Calculating Distance Matrices
### Theory
Distance methods convert a multiple alignment into an $N \times N$ pairwise distance matrix:
- **Fractional Dissimilarity ($p$-distance)**:
  $$p = \frac{\text{Number of mismatching columns}}{\text{Total columns compared (excluding gaps)}}$$
- **Substitution Models**: Jukes-Cantor, Poisson, or BLOSUM62 matrices convert raw $p$-distances into estimates of the true number of substitutions per site, correcting for multiple hits at the same site.


In [ ]:
# Worked Example: Computing pairwise distance matrix with Biopython
from Bio import AlignIO
from Bio.Phylo.TreeConstruction import DistanceCalculator

aln_path = DATA_DIR / "alignments" / "globin_family.aln"
alignment = AlignIO.read(aln_path, "clustal")

calculator = DistanceCalculator("identity")
dm = calculator.get_distance(alignment)

print("=== Globin Pairwise Distance Matrix (Fractional Dissimilarity) ===")
print(dm)


## 5. Concept 3: UPGMA Tree Reconstruction
### Theory
**UPGMA** (Unweighted Pair Group Method with Arithmetic Mean) is a hierarchical clustering algorithm:
1. Find the two closest taxa $A$ and $B$ with minimum distance $d(A, B)$.
2. Join $A$ and $B$ to form a new node $U$. The node height is $d(A, B) / 2$.
3. Compute the distance from node $U$ to every other taxon $C$ as the unweighted arithmetic mean:
   $$d(U, C) = \frac{d(A, C) + d(B, C)}{2}$$
4. Repeat until all taxa are merged into a single rooted tree.

**Assumption:** UPGMA assumes a constant rate of evolution across all lineages (**molecular clock**), producing an **ultrametric tree** where all leaf tips are equidistant from the root.


In [ ]:
# Worked Example: Constructing UPGMA and NJ Trees
from Bio import Phylo
from Bio.Phylo.TreeConstruction import DistanceTreeConstructor
import matplotlib.pyplot as plt

constructor = DistanceTreeConstructor()
upgma_tree = constructor.upgma(dm)
nj_tree = constructor.nj(dm)

# Display ASCII text representation of UPGMA tree
print("=== UPGMA Tree (Assumes Molecular Clock) ===")
Phylo.draw_ascii(upgma_tree)

print("\n=== Neighbor Joining Tree (Allows Variable Rates) ===")
Phylo.draw_ascii(nj_tree)

# Render publication tree plot
fig, ax = plt.subplots(figsize=(8, 5))
Phylo.draw(nj_tree, axes=ax, do_show=False)
ax.set_title("Neighbor Joining Phylogeny of Human Globin Family")
plt.tight_layout()
plt.savefig(OUT_DIR / "globin_nj_tree.png", dpi=100)
plt.show()


## 6. Summary
- Phylogenetic trees model evolutionary divergence from sequence data.
- Orthologues originate from speciation; paralogues originate from gene duplication.
- Distance methods compress alignments into distance matrices.
- UPGMA assumes an equal molecular clock (ultrametric); Neighbor Joining allows lineage-specific rate variation.


## 7. Next Steps
- Reconstruct trees for vertebrate orthologues and insulins in `U07_P_Phylogenetic_Trees.ipynb`.
- Test your tree interpretation skills in `U07_Q_Phylogenetic_Analysis.ipynb`.
